# Suite ALG — The predicate algebra

`Predicates` extends mbse-expressions' Basic with terms about a store's data, each a data class with a builder:
quantifiers over a schema's objects (`Exists`, `Forall`, over an `extent`), predicates and their application, and the
terms of `Distributions` (suite DST).
`Contains` writes Basic's `any` over an adjacency's entries. Mandatory, possible and forbidden links are predicates.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Patterns import Constraints as C, Distributions as D, Predicates as P, Queries as Q, Validators as V
from mbse.Expressions.Framework import Terms
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S
from support import Contact, Phone, book, listed, raises

c, p, x = E.variable("c"), E.variable("p"), E.variable("x")
owns = P.Exists(lambda q: q.symbols({"p": Phone}).requires(P.Contains(c.phones, lambda e: e.phone == p)))
every = P.Forall(lambda q: q.symbols({"c": Contact}).requires(owns))  # every contact has a phone
none = P.Forall(lambda q: q.symbols({"c": Contact}).forbids(owns))  # no contact has one
some = P.Forall(lambda q: q.symbols({"c": Contact}).requires(
    D.Choices.Builder().arms(lambda a: a.weight(0.35).requires(owns),
                             lambda a: a.weight(0.65).requires(E.operation("not", owns))).create()))  # 35% have one

## ALG-01 · The algebra's terms are built by their builders, and checked

In [ ]:
assert isinstance(every, P.OfForall) and every.name == "c" and every.collection.schema == "Contact" and every.body is owns
assert isinstance(owns, P.OfExists) and owns.collection.schema == "Phone"
assert Terms.same(owns.body, E.quantifier("any", "e", E.operation("entries", c, "phones"), E.variable("e").phone.eq(p)).data)
assert Terms.same(none.body, E.operation("not", owns).data) and some.body.arms[0].weight == 0.35
assert P.DIALECT.base is E.DIALECT and P.DIALECT.validate(every, core=True) == [] and P.DIALECT.validate(some) == []
assert [k for k in P.DIALECT.kinds() if k not in E.DIALECT.kinds()] == [
    "extent", "forall", "exists", "predicate", "apply", "set", "choices", "arm", "count", "uniform", "normal", "poisson",
    "geometric", "categorical", "option"]
assert [n for n in P.DIALECT.schemas if n.startswith("Patterns.")][:6] == [
    "Patterns.OfExtent", "Patterns.OfForall", "Patterns.OfExists", "Patterns.Predicate", "Patterns.OfApply", "Patterns.Set"]
assert E.DIALECT.validate(every)[0].startswith("not an expression: ")  # Basic alone cannot check it
assert P.DIALECT.validate(owns) == ["body: collection: argument 0: variable 'c' is not bound"]
# Several symbols are quantified in turn; conditions add up, within them all.
pair = P.Exists(lambda q: q.symbols({"c": Contact}).symbols({"p": "Phone"}).requires(
    P.Contains(c.phones, lambda e: e.phone == p)).forbids(p.has("number")))
assert pair.name == "c" and isinstance(pair.body, P.OfExists) and (pair.body.name, pair.body.collection.schema) == ("p", "Phone")
assert Terms.same(pair.body.body, E.operation("and", owns.body, E.operation("not", p.has("number"))).data)
more = P.OfExists.Builder(owns).requires(p.has("number")).clone()  # from an instance, conditions add to its body
assert Terms.same(more.body, E.operation("and", owns.body, p.has("number")).data) and more.collection is owns.collection
assert P.OfExists.Builder(more).clone().body is more.body and P.Exists(owns) is owns
assert P.Exists(lambda q: q.symbols({"p": Phone})).body is None  # nothing validates yet
assert P.Exists(lambda q: q.requires(True)).name is None  # nor a quantifier without a symbol
numbered = P.Exists(lambda q: q.symbols({"p": Phone}).requires(q.p.has("number")))  # the builder gives its symbols
assert Terms.same(numbered.body, p.has("number").data)
with raises(AttributeError, match="p"):
    P.OfExists.Builder().p  # not declared yet
assert P.OfExists.Builder(more).forbids(True).update() is more and more.body.name == "and"


def written(e):
    return e.phone == p


assert Terms.same(P.Contains(c.phones, written), owns.body)  # a def whose body is one return
assert P.OfExtent.Builder().schema(Contact).create().schema == "Contact"
with raises(TypeError, match="expected a named schema or its name, got "):
    P.Exists(lambda q: q.symbols({"p": S.OfObject.Builder().create()}))
with raises(TypeError, match="expected an exists or a callable taking its builder, got 'p'"):
    P.Exists("p")
with raises(TypeError, match="expected a forall or a callable taking its builder, got 'p'"):
    P.Forall("p")
for adjacency in (c, E.operation("get", c, 1), c.has("phones"), E.operation("get", c, "phones", 1)):
    with raises(TypeError, match="Contains expects an object's adjacency, such as c.phones"):
        P.Contains(adjacency, written)
for condition in (lambda: True, lambda e, f: True, lambda *e: True, 3):
    with raises(TypeError, match="Contains expects a function of one entry"):
        P.Contains(c.phones, condition)

## ALG-02 · The algebra is evaluated over the store

In [ ]:
store = book()
ann = store.Contact().name("ann").age(30).phones(lambda e: e.phone(lambda q: q.number("1"))).create()
bob = store.Contact().name("bob").create()
listed(store, ann, bob)
evaluate = P.Evaluator(store)
assert evaluate(every) is False and evaluate(none) is False and evaluate(some) is True  # bob has none
assert evaluate(owns, {"c": ann}) is True and evaluate(owns, {"c": bob}) is False and evaluate(owns, {"c": None}) is None
assert evaluate(P.Forall(lambda q: q.symbols({"c": Contact}).requires(E.operation("or", owns, c.has("age").not_())))) is True
assert evaluate(P.Exists(lambda q: q.symbols({"c": Contact}).requires(c.name.eq("bob")).forbids(owns))) is True
assert evaluate(P.OfExtent("Contact")) == (ann, bob)
listed(store, store.Contact().name("cy").create())
assert len(evaluate(P.OfExtent("Contact"))) == 2 and len(P.Evaluator(store)(P.OfExtent("Contact"))) == 3  # read once each
with raises(AttributeError, match="no schema registered as 'Nope'"):
    evaluate(P.OfExtent("Nope"))
assert evaluate(x, {"x": True}) is True

## ALG-03 · Predicates written in the algebra: about the whole store, through links, and as data

In [ ]:
store = book()
ann = store.Contact().name("ann").age(30).phones(lambda e: e.phone(lambda q: q.number("1"))).create()
bob = store.Contact().name("bob").create()
listed(store, ann, bob)
EveryoneHasAPhone = P.OfPredicate.Builder().name("EveryoneHasAPhone").requires(every).create()  # no symbols
assert V.Validate(store, [EveryoneHasAPhone])(Contact, ann) == ["the store: 'EveryoneHasAPhone' does not hold"]
query = Q.Scan(store)
assert list(query.select(EveryoneHasAPhone)) == [] and query.explain(EveryoneHasAPhone) == ["first 1 test"]
store.Contact(bob).phones(lambda e: e.phone(lambda q: q.number("2"))).update()
assert list(query.select(EveryoneHasAPhone)) == [{}]  # one match, empty, when it holds
Numbered = P.OfPredicate.Builder().name("Numbered").symbols({"c": Contact, "p": Phone}).requires(
    P.Contains(c.phones, lambda e: e.phone == p)).requires(p.has("number")).create()
assert query.explain(Numbered) == ["c: scan Contact", "p: c.phones to phone, any number, then 2 tests"]
assert [(m["c"].name, m["p"].number) for m in query.select(Numbered)] == [("ann", "1"), ("bob", "2")]
Owner = P.OfPredicate.Builder().name("Owner").symbols({"c": Contact, "p": Phone}).requires(
    P.Contains(p.owners, lambda e: e.owner == c)).create()
assert query.explain(Owner) == ["p: scan Phone", "c: p.owners to owner, at most one, then 1 test"]
Unowned = P.OfPredicate.Builder().name("Unowned").symbols({"p": Phone}).forbids(
    P.Exists(lambda q: q.symbols({"c": Contact}).requires(P.Contains(c.phones, lambda e: e.phone == p)))).create()
assert list(query.select(Unowned)) == [] and len(list(query.select(Unowned, unknown=True))) == 0
# A relation of more than two links: the condition names the link.
Assigned = S.OfRelation.Builder().name("Assigned").links("desk", "person", "seat").create()
Person = S.OfObject.Builder().name("Person").ref().relations(lambda r: r.name("desks").of(Assigned).me("person")).create()
Seat = S.OfObject.Builder().name("Seat").ref().relations(lambda r: r.name("desks").of(Assigned).me("seat")).create()
Desk = S.OfObject.Builder().name("Desk").ref().singleton("office.Desk").relations(
    lambda r: r.name("assigned").of(Assigned).me("desk")).create()
office = Proxies.OfStore()
for schema in (Assigned, Person, Seat, Desk):
    office.register(schema)
desk, pam, seat = office.singleton("office.Desk"), office.Person().create(), office.Seat().create()
office.Desk(desk).assigned(lambda e: e.person(pam).seat(seat)).update()
d = E.variable("d")
Seated = P.OfPredicate.Builder().name("Seated").symbols({"d": Desk, "x": Person}).requires(
    P.Contains(d.assigned, lambda e: e.person == x)).create()
assert Q.Scan(office).explain(Seated) == ["d: scan Desk", "x: d.assigned to person, any number, then 1 test"]
assert list(Q.Scan(office).select(Seated)) == [{"d": desk, "x": pam}]
predicates = C.OfSet.Builder().predicates(EveryoneHasAPhone, Numbered, lambda b: b.name("Possible").requires(some)).create()
schemas = C.OfStore(store)
text = JSON.ToJSON(schemas).Reachable(C.OfSet.Schema, predicates)
assert '"$schema": "Patterns.OfForall"' in text and '"$schema": "Patterns.OfChoices"' in text
copy = JSON.FromJSON(schemas).Reachable(C.OfSet.Schema, text)
assert all(Terms.same(a.rule, b.rule) for a, b in zip(copy.predicates, predicates.predicates))
assert copy.predicates[2].rule.body.arms[0].weight == 0.35